In [1]:
import pandas as pd
from pathlib import Path

proc = Path("..") / "data" / "processed"
e = pd.read_parquet(proc / "empreitadas.parquet")
atos = pd.read_parquet(proc / "modificacoes_atos.parquet")

janela = e[e["ano_celebracao"].between(2019, 2022) & e["modificado"]]
print(f"Empreitadas modificadas 2019-2022: {len(janela)}")

Empreitadas modificadas 2019-2022: 7241


In [2]:
# 1. Como o prazo final se compara ao original nos contratos modificados
razao = janela["prazo_final_mod"] / janela["prazoExecucao"]
print("Prazo final ÷ prazo original:")
print(f"  menor que o original: {(razao < 1).mean():.0%}")
print(f"  igual ao original:    {(razao == 1).mean():.0%}")
print(f"  maior (prorrogado):   {(razao > 1).mean():.0%}")

# 2. A última modificação "esquece" prorrogações anteriores?
#    Comparamos o prazo do último ato com o MAIOR prazo já registado no contrato
maior = atos.groupby("idcontrato")["modifPrazoExecucao"].max().rename("prazo_maximo")
j = janela.merge(maior, on="idcontrato")
esquece = j["prazo_maximo"] > j["prazo_final_mod"]
print(f"\nContratos em que o último ato tem prazo MENOR que um ato anterior: {esquece.mean():.0%} ({esquece.sum()})")

# 3. Os contratos com prorrogação: quanto foi prorrogado
prorrog = janela[razao > 1]
print(f"\nProrrogação nos contratos prorrogados (razão prazo final ÷ original):")
print((prorrog["prazo_final_mod"] / prorrog["prazoExecucao"]).describe(percentiles=[.25, .5, .75, .9]).round(2))

Prazo final ÷ prazo original:
  menor que o original: 2%
  igual ao original:    53%
  maior (prorrogado):   45%

Contratos em que o último ato tem prazo MENOR que um ato anterior: 0% (0)

Prorrogação nos contratos prorrogados (razão prazo final ÷ original):
count    3279.00
mean        1.69
std         1.76
min         1.00
25%         1.19
50%         1.39
75%         1.78
90%         2.44
max        88.50
dtype: float64
